In [ ]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier

nltk.download('punkt')          # токенизатор
nltk.download('stopwords')

train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

train_df = train_df.dropna(subset=['label'])
train_df = train_df[train_df['label'].astype(str).str.strip().isin(['0', '1'])]
train_df['label'] = train_df['label'].astype(int)

stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    text = text.lower()   # К нижнему регистру
    text = re.sub(r'[^a-z\s]', '', text)   # Удаляем всё, кроме букв и пробелов
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word not in stop_words]   # Удаляем стоп-слова
    return ' '.join(tokens)   # Склеиваем обратно в строку

train_df['combined_text'] = train_df['title'].fillna('') + ' ' + train_df['text'].fillna('')
test_df['combined_text'] = test_df['title'].fillna('') + ' ' + test_df['text'].fillna('')

train_df['cleaned_text'] = train_df['combined_text'].apply(preprocess_text)
test_df['cleaned_text'] = test_df['combined_text'].apply(preprocess_text)

tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=14000)
# max_features=4000, ngram_range=(1, 2), результат: 0.98434
# max_features=5000, ngram_range=(1, 1), результат: 0.98365
# max_features=5000, ngram_range=(1, 2), результат: 0.98516
# max_features=5000, ngram_range=(1, 3), результат: 0.98434
# max_features=5500, ngram_range=(1, 2), результат: 0.98457
# max_features=7000, ngram_range=(1, 2), результат: 0.98480
# max_features=15000, ngram_range=(1, 2), результат: 0.98527
# max_features=14000, ngram_range=(1, 2), результат: 0.98528

X = tfidf.fit_transform(train_df['cleaned_text'])
y = train_df['label']
X_test = tfidf.transform(test_df['cleaned_text'])

model = XGBClassifier(eval_metric='logloss')
params = {
    'n_estimators': [500],          # 400 - 500 - 600, резульаты: 0.98504 - 0.98516 - 0.98492 (max_depth = 5, learning_rate = 0.1)
    'max_depth': [5],               # 4 - 5 - 6, результаты: 0.98480 - 0.98516 - 0.98331 (n_estimators = 500, learning_rate = 0.1)
    'learning_rate': [0.1]          # 0.05 - 0.1 - 0.15, результаты: 0.98420 - 0.98516 - 0.98424 (n_estimators = 500, max_depth = 5)
}
grid_search = GridSearchCV(model, params, cv=3, scoring='accuracy', n_jobs=-1)
grid_search.fit(X, y)

print(f"Params: {grid_search.best_params_}")
print(f"CV Accuracy: {grid_search.best_score_:.4f}")

test_pred = grid_search.best_estimator_.predict(X_test)
pd.DataFrame({'id': test_df['id'], 'label': test_pred}).to_csv('submission.csv', index=False)
print("Submission file 'submission.csv' created successfully.")

[nltk_data] Downloading package punkt to C:\Users\Slava
[nltk_data]     Bek\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\Slava
[nltk_data]     Bek\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Params: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 500}
CV Accuracy: 0.9832
Submission file 'submission.csv' created successfully.


In [15]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import StackingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score

nltk.download('punkt')
nltk.download('stopwords')

train_df = pd.read_csv("train.csv", on_bad_lines='skip')
test_df = pd.read_csv("test.csv",  on_bad_lines='skip')

train_df = train_df.loc[:, ~train_df.columns.str.contains(r"^Unnamed")]
test_df = test_df.loc[:,  ~test_df.columns.str.contains(r"^Unnamed")]

train_df = train_df.dropna(subset=['label'])
train_df = train_df[train_df['label'].astype(str).str.strip().isin(['0', '1'])]
train_df['label'] = train_df['label'].astype(int)

stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words]
    return " ".join(tokens)

train_df['combined_text'] = train_df['title'].fillna('') + ' ' + train_df['text'].fillna('')
test_df['combined_text'] = test_df['title'].fillna('') + ' ' + test_df['text'].fillna('')

train_df['cleaned_text'] = train_df['combined_text'].apply(preprocess_text)
test_df['cleaned_text'] = test_df['combined_text'].apply(preprocess_text)

tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=14000)
X = tfidf.fit_transform(train_df['cleaned_text'])
y = train_df['label']
X_test = tfidf.transform(test_df['cleaned_text'])

estimate = [
    ('xgb', XGBClassifier(n_estimators=500, max_depth=5, learning_rate=0.1, eval_metric='logloss')),
    ('dt', DecisionTreeClassifier(random_state=42)),
]

stack = StackingClassifier(estimators=estimate, final_estimator=LogisticRegression(max_iter=1000), cv=3, n_jobs=-1)

cv_scores = cross_val_score(stack, X, y, cv=3, scoring='accuracy', n_jobs=-1)
print(f"CV Accuracy: {cv_scores.mean():.4f}")

stack.fit(X, y)

test_pred = grid_search.best_estimator_.predict(X_test)
pd.DataFrame({'id': test_df['id'], 'label': test_pred}).to_csv('submission.csv', index=False)
print("Submission file 'submission.csv' created successfully.")

[nltk_data] Downloading package punkt to C:\Users\Slava
[nltk_data]     Bek\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\Slava
[nltk_data]     Bek\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


CV Accuracy: 0.9832
Submission file 'submission.csv' created successfully.
